In [1]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
data = pd.read_csv("jopacc_system_month_november_2022_to_june_2026.csv")
data.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_usd_transaction_value,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity


In [3]:
data.columns

Index(['report_month', 'report_year', 'report_quarter', 'system',
       'users_total', 'transactions_total', 'transaction_value_jod',
       'user_jordanian_count', 'user_non_jordanian_count', 'user_male_count',
       'user_female_count', 'user_age_under_30_count', 'user_age_31_50_count',
       'user_age_51_plus_count', 'customer_individual_count',
       'customer_legal_entity_count', 'wallet_individual_count',
       'wallet_commercial_count', 'user_existing_count', 'user_new_count',
       'tx_purchases_count', 'tx_money_transfers_count', 'tx_cash_out_count',
       'tx_cash_in_count', 'value_purchases_jod', 'value_money_transfers_jod',
       'value_cash_out_jod', 'value_cash_in_jod', 'on_us_money_transfer_count',
       'off_us_money_transfer_count', 'on_us_money_transfer_value_jod',
       'off_us_money_transfer_value_jod', 'billers_total', 'services_total',
       'ef_cash_payment_count', 'ef_digital_payment_count',
       'ef_cash_payment_value_jod', 'ef_digital_payment_valu

In [4]:
print("Number of rows:", data.shape[0])
print("Number of columns:", data.shape[1])

print("Missing values:", data.isnull().sum().sum())
print("Duplicated rows:", data.duplicated().sum())

Number of rows: 214
Number of columns: 50
Missing values: 0
Duplicated rows: 0


In [5]:
data["report_month"].dtype

<StringDtype(na_value=nan)>

In [6]:
data["report_month"] = pd.to_datetime(data["report_month"],format="%Y-%m")
data["report_month_number"] = data["report_month"].dt.month
data = data.sort_values("report_month").reset_index(drop=True)
data.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,report_month_number
0,2022-11-01,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0,0.0,0.0,Low Activity,11
1,2022-11-01,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0,0.0,0.0,Low Activity,11
2,2022-11-01,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0,0.0,0.0,Low Activity,11
3,2022-12-01,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0,0.0,0.0,Low Activity,12
4,2022-12-01,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0,0.0,0.0,Low Activity,12


In [7]:
USD_TO_JOD = 0.709
EUR_TO_JOD = 0.770
GBP_TO_JOD = 0.900

data["ach_usd_value_jod"] = (data["ach_usd_transaction_value"] * USD_TO_JOD)
data["ach_eur_value_jod"] = (data["ach_eur_transaction_value"] * EUR_TO_JOD)
data["ach_gbp_value_jod"] = (data["ach_gbp_transaction_value"] * GBP_TO_JOD)

data[
    [
        "ach_usd_value_jod",
        "ach_eur_value_jod",
        "ach_gbp_value_jod"
    ]
].head()

,ach_usd_value_jod,ach_eur_value_jod,ach_gbp_value_jod
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0


In [8]:
categorical_features = data.select_dtypes(include=["object", "category"]).columns

print("Categorical features:")
print(categorical_features.tolist())

Categorical features:
['report_quarter', 'system', 'activity_level']


C:\Users\NTC\AppData\Local\Temp\ipykernel_12060\422581742.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = data.select_dtypes(include=["object", "category"]).columns


In [9]:
data["report_quarter"] = (data["report_quarter"].str.replace("Q", "", regex=False).astype(int))

activity_encoder = LabelEncoder()

data["activity_level"] = activity_encoder.fit_transform(data["activity_level"])

system_encoder = OneHotEncoder(sparse_output=False)
system_encoded = system_encoder.fit_transform(data[["system"]])

system_encoded_df = pd.DataFrame(system_encoded,
    columns=system_encoder.get_feature_names_out(["system"]),
    index=data.index
)

data = pd.concat([data.drop(columns=["system"]), system_encoded_df],axis=1)
data.head()

,report_month,report_year,report_quarter,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,user_female_count,...,activity_level,report_month_number,ach_usd_value_jod,ach_eur_value_jod,ach_gbp_value_jod,system_ACH,system_CliQ,system_ECCU,system_JoMoPay,system_eFAWATEERcom
0,2022-11-01,2022,4,523000,985000,175000000,494200,24100,342500,171500,...,1,11,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2022-11-01,2022,4,2020000,1260000,155000000,1760000,250000,1230000,770000,...,1,11,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2022-11-01,2022,4,3580000,3630000,848000000,0,0,0,0,...,1,11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2022-12-01,2022,4,561000,1180000,209000000,529800,26300,365000,186200,...,1,12,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,2022-12-01,2022,4,2050000,1510000,188000000,1780000,250000,1230000,800000,...,1,12,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [10]:
system_features = system_encoder.get_feature_names_out(["system"]).tolist()
system_features

['system_ACH',
 'system_CliQ',
 'system_ECCU',
 'system_JoMoPay',
 'system_eFAWATEERcom']

In [11]:
joblib.dump(system_encoder, "pkl_files/system_encoder.pkl")

['pkl_files/system_encoder.pkl']

In [12]:
selected_numeric_features = [
    "report_year",
    "report_month_number",
    "users_total",
    "transactions_total",
    "customer_legal_entity_count",
    "wallet_individual_count",
    "user_new_count",
    "tx_purchases_count",
    "tx_money_transfers_count",
    "ef_digital_payment_count",
    "ach_jod_transaction_count",
    "returned_cheque_count"
]

selected_features = (selected_numeric_features + system_features)

print("Number of selected features:",len(selected_features))

Number of selected features: 17


In [13]:
X_reg = data[selected_features].copy()
y_reg = data["transaction_value_jod"].copy()

X_clf = data[selected_features].copy()
y_clf = data["activity_level"].copy()

print("Regression features:", X_reg.shape)
print("Classification features:", X_clf.shape)

Regression features: (214, 17)
Classification features: (214, 17)


In [14]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg,y_reg,test_size=0.2,random_state=42)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf,y_clf,test_size=0.2,random_state=42,stratify=y_clf)

print("Regression:")
print("X_train:", X_train_reg.shape)
print("X_test:", X_test_reg.shape)

print("\nClassification:")
print("X_train:", X_train_clf.shape)
print("X_test:", X_test_clf.shape)

Regression:
X_train: (171, 17)
X_test: (43, 17)

Classification:
X_train: (171, 17)
X_test: (43, 17)


In [15]:
scaler_reg = StandardScaler()

X_train_reg[selected_numeric_features] = scaler_reg.fit_transform(X_train_reg[selected_numeric_features])
X_test_reg[selected_numeric_features] = scaler_reg.transform(X_test_reg[selected_numeric_features])

In [16]:
scaler_clf = StandardScaler()

X_train_clf[selected_numeric_features] = scaler_clf.fit_transform(X_train_clf[selected_numeric_features])
X_test_clf[selected_numeric_features] = scaler_clf.transform(X_test_clf[selected_numeric_features])

In [18]:
joblib.dump(
    {
        "X_train": X_train_clf,
        "X_test": X_test_clf,
        "y_train": y_train_clf,
        "y_test": y_test_clf,
        "scaler": scaler_clf,
        "activity_encoder": activity_encoder,
        "system_encoder": system_encoder,
        "selected_features": selected_features,
        "numeric_features": selected_numeric_features
    },
    "pkl_files/preprocessed_classification.pkl"
)

joblib.dump(
    {
        "X_train": X_train_reg,
        "X_test": X_test_reg,
        "y_train": y_train_reg,
        "y_test": y_test_reg,
        "scaler": scaler_reg,
        "system_encoder": system_encoder,
        "selected_features": selected_features,
        "numeric_features": selected_numeric_features
    },
    "pkl_files/preprocessed_regression.pkl"
)
print("Preprocessed data saved successfully")

Preprocessed data saved successfully


In [19]:
import os
from model import BASE_DIR

_classification_preprocessing = joblib.load(os.path.join(BASE_DIR, "pkl_files/preprocessed_classification.pkl"))
_activity_encoder = _classification_preprocessing["activity_encoder"]

print(_activity_encoder.classes_)

for number, class_name in enumerate(_activity_encoder.classes_):
    print(number, "=", class_name)

['High Activity' 'Low Activity' 'Normal Activity']
0 = High Activity
1 = Low Activity
2 = Normal Activity
